# Daily minimum-risk portfolio of the 23 tickers

Every day the weights `w` are chosen to minimise the forecast variance of the portfolio, `w' Σ w`, with `Σ` built from that day's forecasts:

- `D`: the model's variance forecast of every ticker (here the long model `xgb_long_drv_har`, trained once on the 23 tickers, pooled; it predicts the change of the realized variance),
- `R`: the correlation of the EWMA covariance of that day,
- `Σ = D R D`.

Constraints: the weights sum to 1 and none is negative (long only, fully invested). The weights of a day use only forecasts made before that day; the portfolio return of the day is `w · r` (the day's returns).
Nothing is trained here: the forecasts are read from the saved run.

In [1]:
import contextlib
import io

import numpy as np
import pandas as pd
from scipy.optimize import minimize

import run_store as rs

pd.set_option('display.width', 200)
TICKERS = ['AAPL', 'MSFT', 'NVDA', 'AVGO', 'GOOGL', 'META', 'NFLX', 'AMZN', 'HD', 'MCD', 'PG', 'KO', 'WMT', 'XOM', 'CVX', 'JPM', 'GS', 'BAC', 'V', 'JNJ', 'UNH', 'LLY', 'MRK']
WF = 'from2016-03-01_to2026-06-30_train756bars_forecast21bars_23tickers_d93e1cd5'   # the 23-ticker walk-forward of train_23_experiment.py
runs = {r.name: r for r in rs.load_runs(train_bars=756, forecast_bars=21) if r.walk_forward_id == WF}
bank = runs['xgb_long_drv_har'].ewma_bank       # per forecast day: the model's ticker variances (sigma_*), the EWMA covariance (cov_*), the day's returns (return_*)
bank[[f'sigma_{t}' for t in TICKERS] + [f'return_{t}' for t in TICKERS]].head()

,sigma_AAPL,sigma_MSFT,sigma_NVDA,sigma_AVGO,sigma_GOOGL,sigma_META,sigma_NFLX,sigma_AMZN,sigma_HD,sigma_MCD,sigma_PG,sigma_KO,sigma_WMT,sigma_XOM,sigma_CVX,sigma_JPM,sigma_GS,sigma_BAC,sigma_V,sigma_JNJ,sigma_UNH,sigma_LLY,sigma_MRK,return_AAPL,return_MSFT,return_NVDA,return_AVGO,return_GOOGL,return_META,return_NFLX,return_AMZN,return_HD,return_MCD,return_PG,return_KO,return_WMT,return_XOM,return_CVX,return_JPM,return_GS,return_BAC,return_V,return_JNJ,return_UNH,return_LLY,return_MRK
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2019-03-04,0.000073,0.000065,0.000217,0.000150,0.000106,0.000099,0.000160,0.000094,0.000128,0.000072,0.000051,0.000054,0.000247,0.000101,0.000140,0.000100,0.000145,0.000106,0.000062,0.000051,0.000192,0.000261,0.000069,0.005029,-0.002399,0.002109,0.007846,0.004266,0.031365,-0.017575,0.014620,-0.007344,-0.024102,0.004368,0.005950,-0.000817,0.003875,0.000738,-0.002298,-0.011201,-0.009553,-0.010102,0.000940,-0.041153,-0.010711,-0.003429
2019-03-05,0.000139,0.000127,0.000302,0.000153,0.000162,0.000241,0.000341,0.000238,0.000116,0.000089,0.000057,0.000064,0.000090,0.000126,0.000114,0.000112,0.000212,0.000160,0.000092,0.000052,0.000429,0.000501,0.000093,-0.001820,-0.004989,-0.001658,0.007712,0.013672,0.023242,0.009287,-0.002205,0.001033,0.004208,0.002225,-0.001095,0.005008,-0.001494,0.009580,-0.000768,0.000102,0.001722,-0.000068,0.002094,0.022922,0.008081,0.004056
2019-03-06,0.000084,0.000082,0.000242,0.000152,0.000140,0.000227,0.000326,0.000145,0.000108,0.000056,0.000060,0.000053,0.000098,0.000099,0.000091,0.000085,0.000148,0.000132,0.000082,0.000049,0.000295,0.000291,0.000063,-0.005754,0.000448,-0.028559,-0.020433,-0.003635,0.007299,0.014987,-0.013873,0.002446,0.003750,-0.001615,-0.003290,-0.000813,-0.011348,0.001947,-0.003746,-0.006428,-0.008253,-0.000946,0.002306,-0.009775,-0.014087,-0.011506
2019-03-07,0.000081,0.000087,0.000322,0.000207,0.000124,0.000227,0.000187,0.000124,0.000062,0.000061,0.000043,0.000049,0.000086,0.000123,0.000176,0.000093,0.000140,0.000104,0.000056,0.000042,0.000217,0.000313,0.000099,-0.011574,-0.012170,-0.018349,-0.021522,-0.012095,-0.019593,-0.019493,-0.025765,-0.010518,-0.008295,-0.003841,-0.003741,-0.008243,0.011100,-0.012871,-0.007231,-0.010116,-0.010056,-0.006630,-0.006111,-0.009913,-0.002684,-0.003839
2019-03-08,0.000099,0.000104,0.000419,0.000277,0.000132,0.000177,0.000264,0.000136,0.000087,0.000062,0.000073,0.000054,0.000088,0.000118,0.000129,0.000100,0.000237,0.000154,0.000072,0.000056,0.000239,0.000202,0.000095,0.002376,0.001087,0.009246,-0.004971,-0.000765,0.002779,-0.008508,-0.003167,-0.007013,-0.005650,-0.001421,-0.009717,0.001436,-0.014346,-0.002624,0.000388,0.012813,0.003503,0.003542,-0.001302,0.006319,0.002850,-0.008080


## 1. Σ of a day and the weights that minimise `w' Σ w`

In [2]:
def sigma_matrices(bank):
    """Sigma_d = D R D for every day: D = sqrt of the model's ticker variance, R = the correlation of that day's EWMA covariance."""
    d = np.sqrt(bank[[f'sigma_{t}' for t in TICKERS]].to_numpy())
    cov = np.stack([bank[[f'cov_{i}_{j}' for j in TICKERS]].to_numpy() for i in TICKERS], axis=1)      # day x i x j
    diag = np.sqrt(np.einsum('dii->di', cov))
    corr = cov / (diag[:, :, None] * diag[:, None, :])
    return d[:, :, None] * corr * d[:, None, :]


def min_variance_weights(sigma, start):
    """min w' S w  s.t. sum w = 1, w >= 0.  SLSQP; S is divided by its mean diagonal so that the numbers are about 1."""
    s = sigma / (np.trace(sigma) / len(sigma))
    res = minimize(lambda w: w @ s @ w, start, jac=lambda w: 2 * s @ w, bounds=[(0, 1)] * len(start),
                   constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1, 'jac': lambda w: np.ones_like(w)}],
                   method='SLSQP', options={'ftol': 1e-12, 'maxiter': 200})
    w = np.clip(res.x, 0, None)
    return w / w.sum()


sigmas = sigma_matrices(bank)
day = pd.Timestamp('2024-03-15')
k = bank.index.get_loc(day)
vol = np.sqrt(np.diag(sigmas[k]) * 252)
corr = pd.DataFrame(sigmas[k] / np.outer(np.sqrt(np.diag(sigmas[k])), np.sqrt(np.diag(sigmas[k]))), index=TICKERS, columns=TICKERS)
print(f'{day.date()}: forecast annualised volatility per ticker (%):', {t: float(v) for t, v in zip(TICKERS, (vol * 100).round(1))})
corr.round(2)

2024-03-15: forecast annualised volatility per ticker (%): {'AAPL': 17.8, 'MSFT': 21.2, 'NVDA': 49.7, 'AVGO': 29.6, 'GOOGL': 20.7, 'META': 30.6, 'NFLX': 26.7, 'AMZN': 22.2, 'HD': 17.5, 'MCD': 16.8, 'PG': 12.0, 'KO': 11.3, 'WMT': 13.8, 'XOM': 16.0, 'CVX': 17.2, 'JPM': 18.0, 'GS': 23.5, 'BAC': 22.5, 'V': 14.7, 'JNJ': 13.6, 'UNH': 17.3, 'LLY': 24.8, 'MRK': 19.7}


,AAPL,MSFT,NVDA,AVGO,GOOGL,META,NFLX,AMZN,HD,MCD,PG,KO,WMT,XOM,CVX,JPM,GS,BAC,V,JNJ,UNH,LLY,MRK
AAPL,1.00,0.45,-0.07,0.16,0.55,0.03,0.28,0.23,0.07,0.19,-0.03,0.07,-0.17,0.01,0.18,-0.33,-0.13,-0.25,0.35,0.29,0.33,-0.09,0.31
MSFT,0.45,1.00,0.46,0.50,0.55,0.52,0.54,0.79,0.31,-0.03,0.01,0.05,-0.02,-0.01,0.10,-0.14,0.04,-0.04,0.40,-0.06,0.14,0.41,0.10
NVDA,-0.07,0.46,1.00,0.63,0.04,0.53,0.42,0.57,0.42,-0.00,0.03,-0.18,0.23,-0.23,-0.19,0.31,0.13,0.12,0.34,0.10,-0.04,0.63,0.13
AVGO,0.16,0.50,0.63,1.00,0.07,0.45,0.52,0.51,0.32,0.10,0.00,-0.27,-0.02,-0.06,-0.02,-0.14,0.02,-0.06,0.12,-0.02,0.07,0.55,0.19
GOOGL,0.55,0.55,0.04,0.07,1.00,0.14,0.21,0.42,0.11,-0.13,0.04,0.19,-0.21,0.20,0.34,-0.14,-0.14,-0.00,0.17,0.05,0.20,-0.11,0.10
META,0.03,0.52,0.53,0.45,0.14,1.00,0.41,0.78,0.29,0.03,-0.08,-0.21,0.06,-0.04,0.15,0.19,0.20,0.09,0.22,-0.03,-0.04,0.41,0.20
NFLX,0.28,0.54,0.42,0.52,0.21,0.41,1.00,0.47,0.26,-0.07,-0.05,-0.24,-0.12,0.12,0.16,0.04,-0.02,0.11,0.27,0.03,0.03,0.41,0.12
AMZN,0.23,0.79,0.57,0.51,0.42,0.78,0.47,1.00,0.36,-0.12,-0.05,-0.09,-0.06,-0.04,0.15,0.03,0.11,-0.05,0.32,-0.13,0.02,0.49,0.16
HD,0.07,0.31,0.42,0.32,0.11,0.29,0.26,0.36,1.00,-0.12,0.11,0.18,0.13,0.10,0.12,0.46,0.49,0.27,0.35,0.31,-0.15,0.31,0.39
MCD,0.19,-0.03,-0.00,0.10,-0.13,0.03,-0.07,-0.12,-0.12,1.00,0.12,0.01,0.17,-0.07,-0.07,0.05,-0.01,0.08,0.12,0.39,0.13,-0.22,0.17


In [3]:
w_day = min_variance_weights(sigmas[k], np.full(len(TICKERS), 1 / len(TICKERS)))
equal = np.full(len(TICKERS), 1 / len(TICKERS))
annual_vol = lambda weights: round(float(np.sqrt(weights @ sigmas[k] @ weights * 252)) * 100, 2)
print(f'forecast volatility of the portfolio (% / year): minimum risk {annual_vol(w_day)}, equal weight {annual_vol(equal)}')
pd.DataFrame({'weight (%)': (w_day * 100).round(1), 'equal weight (%)': equal * 100}, index=TICKERS)

forecast volatility of the portfolio (% / year): minimum risk 5.87, equal weight 9.45


,weight (%),equal weight (%)
AAPL,9.3,4.347826
MSFT,0.0,4.347826
NVDA,0.0,4.347826
AVGO,0.0,4.347826
GOOGL,3.6,4.347826
META,0.0,4.347826
NFLX,0.0,4.347826
AMZN,0.0,4.347826
HD,0.0,4.347826
MCD,12.4,4.347826


## 2. Every day: the weights as percentages

In [4]:
w = np.full(len(TICKERS), 1 / len(TICKERS))
rows = []
for s in sigmas:
    w = min_variance_weights(s, w)      # the solution of the day before is the starting point; the optimum is unique
    rows.append(w)
weights = pd.DataFrame(rows, index=bank.index, columns=TICKERS)
returns = bank[[f'return_{t}' for t in TICKERS]]
weights_pct = (weights * 100).round(2)
weights_pct.insert(0, 'forecast vol (% / year)', (np.sqrt(np.einsum('di,dij,dj->d', weights.to_numpy(), sigmas, weights.to_numpy()) * 252) * 100).round(2))
weights_pct['day return (%)'] = ((weights * returns.to_numpy()).sum(axis=1) * 100).round(3)
weights_pct.index.name = 'Date'
print(len(weights_pct), 'days; the first and last rows (the full table is `weights_pct`)')
weights_pct

1842 days; the first and last rows (the full table is `weights_pct`)


,forecast vol (% / year),AAPL,MSFT,NVDA,AVGO,GOOGL,META,NFLX,AMZN,HD,MCD,PG,KO,WMT,XOM,CVX,JPM,GS,BAC,V,JNJ,UNH,LLY,MRK,day return (%)
Date,,,,,,,,,,,,,,,,,,,,,,,,,
2019-03-04,6.73,1.50,0.00,0.00,0.00,0.00,14.14,0.00,0.00,0.00,21.52,12.14,28.88,2.09,0.00,0.00,0.00,0.00,5.00,3.76,9.25,1.72,0.00,0.00,0.008
2019-03-05,6.98,0.00,0.00,0.00,1.84,0.00,6.92,0.00,0.00,0.00,19.97,8.49,27.76,14.65,0.00,0.00,0.00,0.00,2.40,0.00,17.97,0.00,0.00,0.00,0.363
2019-03-06,6.38,8.82,0.00,0.00,0.00,0.00,4.80,0.00,0.00,0.00,28.10,3.69,30.62,11.76,1.07,0.00,0.00,0.00,0.00,0.00,11.13,0.00,0.00,0.00,-0.013
2019-03-07,6.15,6.21,0.00,0.00,0.00,0.00,3.43,0.00,0.00,0.00,21.19,10.62,28.19,11.39,0.00,0.00,0.00,0.00,2.68,1.56,14.72,0.00,0.00,0.00,-0.682
2019-03-08,6.64,4.47,0.00,0.00,0.00,0.00,7.09,0.00,0.00,0.00,26.91,0.00,33.31,14.26,7.95,0.00,0.00,0.00,0.00,0.00,6.00,0.00,0.00,0.00,-0.547
2019-03-11,6.62,8.75,0.00,0.00,0.00,0.00,3.63,0.00,0.00,0.00,24.05,5.27,32.88,15.56,6.80,0.00,0.00,1.30,0.00,0.00,1.69,0.07,0.00,0.00,1.834
2019-03-12,7.08,0.00,0.00,0.00,0.00,0.00,5.23,0.00,0.00,0.00,18.40,7.41,25.82,16.72,10.98,0.00,0.00,6.26,0.00,0.00,1.72,0.00,7.46,0.00,0.074
2019-03-13,6.66,0.00,0.00,0.00,0.00,0.00,9.13,0.00,0.00,0.00,19.70,0.00,22.06,24.69,8.90,0.00,0.00,3.88,0.00,0.00,0.05,0.00,11.58,0.00,0.516
2019-03-14,6.64,0.00,0.00,0.00,0.00,0.00,15.48,0.00,0.00,0.00,28.89,0.00,27.94,10.08,4.98,0.00,0.00,3.42,0.00,0.00,4.74,0.00,4.47,0.00,-0.384


## 3. What the weights look like (sorted by the mean weight)

In [5]:
summary = pd.DataFrame({
    'mean weight (%)': (weights.mean() * 100).round(1),
    'min weight (%)': (weights.min() * 100).round(1),
    'max weight (%)': (weights.max() * 100).round(1),
    'days with 0 weight (%)': ((weights < 1e-6).mean() * 100).round(1),
    'days with weight > 50% (%)': ((weights > 0.5).mean() * 100).round(1),
}).sort_values('mean weight (%)', ascending=False)
print('mean daily turnover (sum of |change of the weights|, % of the portfolio):', round(float(weights.diff().abs().sum(axis=1).mean() * 100), 1))
print('days with all the weight in one ticker:', int((weights > 0.999).any(axis=1).sum()))
summary

mean daily turnover (sum of |change of the weights|, % of the portfolio): 59.4
days with all the weight in one ticker: 3


,mean weight (%),min weight (%),max weight (%),days with 0 weight (%),days with weight > 50% (%)
WMT,10.7,0.0,59.6,20.1,0.5
JNJ,10.1,0.0,65.4,27.5,0.4
KO,9.9,0.0,61.3,33.4,0.2
PG,8.6,0.0,82.5,31.8,0.2
MCD,8.3,0.0,64.8,30.3,0.3
CVX,6.0,0.0,38.9,47.8,0.0
MRK,6.0,0.0,49.3,44.7,0.0
XOM,4.5,0.0,31.6,47.0,0.0
UNH,4.4,0.0,43.7,46.4,0.0
LLY,4.3,0.0,40.0,44.1,0.0


## 4. Realized risk against the fixed equal-weight portfolio

Annualised volatility of the daily portfolio return (weights of the day applied to that day's returns), over the whole period and in two parts.

In [6]:
equal_return = returns.mean(axis=1)                                  # 1/23 each, rebalanced daily
minvar_return = (weights * returns.to_numpy()).sum(axis=1)


def period(mask):
    out = {}
    for name, r in (('equal weight', equal_return), ('minimum risk', minvar_return)):
        r = r[mask]
        out[name] = {
            'days': int(mask.sum()), 'volatility (% / year)': round(float(r.std() * np.sqrt(252)) * 100, 1),
            'worst day (%)': round(float(r.min()) * 100, 2), 'mean of the worst 5% days (%)': round(float(r[r <= r.quantile(0.05)].mean()) * 100, 2),
        }
    return pd.DataFrame(out).T.astype({'days': int})


idx = bank.index
pd.concat({
    'whole period': period(np.ones(len(idx), bool)),
    'Covid crash (2020-02-19 .. 2020-06-30)': period((idx >= '2020-02-19') & (idx <= '2020-06-30')),
    'after Covid (2020-07-01 on)': period(idx >= '2020-07-01'),
})

days  volatility (% / year)  worst day (%)  mean of the worst 5% days (%)
whole period                           equal weight  1842                   19.0         -12.31                          -2.80
                                       minimum risk  1842                   14.8          -8.99                          -2.22
Covid crash (2020-02-19 .. 2020-06-30) equal weight    93                   53.3         -12.31                          -8.09
                                       minimum risk    93                   38.2          -8.99                          -5.54
after Covid (2020-07-01 on)            equal weight  1506                   15.5          -5.57                          -2.24
                                       minimum risk  1506                   12.8          -6.03                          -1.89

## 5. Daily returns: minimum risk against equal weight

The realized returns are read from the walk-forward's calendars (`forecast_data` of every window: the returns of its forecast days), not from the bank. The equal-weight return is the calendar's own `portfolio_return` (1/23 each); the minimum-risk return applies the day's weights to the tickers' returns of that day.

In [7]:
DATA_PATH = './data_general/23_ticker_data_2010_2026.csv'
prices = pd.read_csv(DATA_PATH, parse_dates=['Date'], index_col='Date')
setting = rs.load_walk_forward(WF, prices)                       # the setting of the run's walk-forward (the hash of the data is checked)
with contextlib.redirect_stdout(io.StringIO()):                  # the calendars print a warning for every market holiday they skip
    calendars = setting.make_calendars()                         # one calendar per window
forecast_days = pd.concat([c.forecast_data for c in calendars])  # the realized returns of every forecast day
calendar_returns = forecast_days[TICKERS].loc[weights.index]
print('days', len(forecast_days), '| the calendars give the same returns as the bank:', bool(np.allclose(calendar_returns.to_numpy(), returns.to_numpy())))

equal_weight = forecast_days['portfolio_return'].loc[weights.index]                     # the calendar's own portfolio return
minimum_risk = (weights * calendar_returns).sum(axis=1)                                 # the day's weights applied to that day's returns
daily = pd.DataFrame({'equal weight (%)': equal_weight * 100, 'minimum risk (%)': minimum_risk * 100})
daily['difference (%)'] = daily['minimum risk (%)'] - daily['equal weight (%)']
daily.round(3)

days 1842 | the calendars give the same returns as the bank: True


,equal weight (%),minimum risk (%),difference (%)
Date,,,
2019-03-04,-0.259,0.008,0.267
2019-03-05,0.438,0.363,-0.076
2019-03-06,-0.482,-0.013,0.469
2019-03-07,-1.015,-0.682,0.333
2019-03-08,-0.092,-0.547,-0.455
2019-03-11,1.737,1.834,0.097
2019-03-12,0.364,0.074,-0.289
2019-03-13,0.791,0.516,-0.275
2019-03-14,-0.153,-0.384,-0.231


In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REFERENCE_COLOR, MODEL_COLOR = '#9aa0a6', '#1f5fbf'          # the colours of the dashboard: the reference (grey) and the model (blue)
CRASH = ('2020-02-19', '2020-06-30')                          # the Covid crash
WINDOW = 63
rolling_volatility = lambda r: r.rolling(WINDOW).std() * np.sqrt(252) * 100

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07, row_heights=[0.38, 0.38, 0.24],
    subplot_titles=('Daily return (%)', 'Cumulative return (growth of 1, the weights of the day, rebalanced daily)', f'Realized volatility over the last {WINDOW} days (% / year)'),
)
for name, r, color in (('equal weight', equal_weight, REFERENCE_COLOR), ('minimum risk', minimum_risk, MODEL_COLOR)):
    fig.add_trace(go.Scatter(x=r.index, y=r * 100, name=name, legendgroup=name, line=dict(color=color, width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=r.index, y=(1 + r).cumprod(), name=name, legendgroup=name, showlegend=False, line=dict(color=color, width=1.6)), row=2, col=1)
    fig.add_trace(go.Scatter(x=r.index, y=rolling_volatility(r), name=name, legendgroup=name, showlegend=False, line=dict(color=color, width=1.6)), row=3, col=1)

# the daily return as a log return, ln(1 + r), hidden until the button is pressed (the cumulative return and the volatility are the same either way)
n_simple = len(fig.data)
for name, r, color in (('equal weight', equal_weight, REFERENCE_COLOR), ('minimum risk', minimum_risk, MODEL_COLOR)):
    fig.add_trace(go.Scatter(x=r.index, y=np.log1p(r) * 100, name=name, legendgroup=name, visible=False, line=dict(color=color, width=1)), row=1, col=1)
simple_view = [True] * n_simple + [False, False]
log_view = [i not in (0, 3) for i in range(n_simple)] + [True, True]      # traces 0 and 3 are the simple daily returns

fig.add_vrect(x0=CRASH[0], x1=CRASH[1], fillcolor='#ef6c00', opacity=0.10, line_width=0, row='all', col=1)
fig.update_yaxes(ticksuffix='%', row=1, col=1)
fig.update_yaxes(ticksuffix='%', row=3, col=1)
fig.update_xaxes(rangeslider=dict(visible=True, thickness=0.05), row=3, col=1)
fig.update_layout(
    template='plotly_white', hovermode='x unified', height=860,
    legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='right', x=1),
    updatemenus=[
        dict(type='buttons', direction='right', x=0, y=1.10, xanchor='left', yanchor='bottom', buttons=[
            dict(label='daily return: percent', method='restyle', args=[{'visible': simple_view}]),
            dict(label='daily return: log, ln(1 + r)', method='restyle', args=[{'visible': log_view}]),
        ]),
        dict(type='buttons', direction='right', x=0.50, y=1.10, xanchor='left', yanchor='bottom', buttons=[
            dict(label='growth of 1: linear', method='relayout', args=[{'yaxis2.type': 'linear'}]),
            dict(label='growth of 1: log axis', method='relayout', args=[{'yaxis2.type': 'log'}]),
        ]),
    ],
)
fig

Figure(8 traces)

## 6. Checks of the numbers

In [9]:
# the weights of every day are the optimum: the KKT conditions of  min w' S w  s.t. sum w = 1, w >= 0
n = len(TICKERS)
spread, violation = [], []
for s, wd in zip(sigmas, weights.to_numpy()):
    g = 2 * (s / (np.trace(s) / n)) @ wd
    held = wd > 1e-7
    lam = g[held].mean()
    spread.append(np.abs(g[held] - lam).max() / abs(lam))
    violation.append(max(0.0, float((lam - g[~held]).max() / abs(lam))) if (~held).any() else 0.0)
print(f'KKT over {len(sigmas)} days: on the held tickers the gradient differs from its mean by at most {max(spread):.1e} (relative); a ticker with weight 0 breaks the condition by at most {max(violation):.1e}')
n_held = (weights > 0.01).sum(axis=1)
print(f'tickers with a weight above 1% per day: mean {n_held.mean():.1f}, min {n_held.min()}, max {n_held.max()} (of {n}) | the largest weight of the day: mean {weights.max(axis=1).mean():.2f}, max {weights.max().max():.2f}')

KKT over 1842 days: on the held tickers the gradient differs from its mean by at most 1.2e-05 (relative); a ticker with weight 0 breaks the condition by at most 0.0e+00
tickers with a weight above 1% per day: mean 9.7, min 1, max 18 (of 23) | the largest weight of the day: mean 0.27, max 1.00


In [10]:
# the variance the model forecast for the portfolio against what happened: realized r^2 / forecast variance (1 = calibrated)
forecast_variance = pd.DataFrame({
    'equal weight': np.einsum('i,dij,j->d', np.full(n, 1 / n), sigmas, np.full(n, 1 / n)),
    'minimum risk': np.einsum('di,dij,dj->d', weights.to_numpy(), sigmas, weights.to_numpy()),
}, index=bank.index)
realized_variance = pd.DataFrame({'equal weight': equal_return ** 2, 'minimum risk': minvar_return ** 2})
pd.DataFrame({
    'forecast volatility (% / year)': (np.sqrt(forecast_variance.mean() * 252) * 100).round(2),
    'realized r^2 / forecast variance': (realized_variance.mean() / forecast_variance.mean()).round(2),
})

,forecast volatility (% / year),realized r^2 / forecast variance
equal weight,15.66,1.47
minimum risk,10.58,1.98


## Caveats seen in this prototype

- The weights move a lot from day to day (the turnover of section 3): there is no cost for trading and no limit on the change.
- There is no cap on a weight: sections 3 and 6 show how many tickers the portfolio holds and how large the largest weight gets.
- The optimiser looks for the portfolio the forecast covariance says is the safest, so it picks the places where the forecast under-estimates the risk: compare the two numbers of the last table (realized `r²` over the forecast variance, 1 = calibrated). A VaR computed from this `Σ` alone would be too low if the minimum-risk portfolio is further from 1 than the equal-weight one.
- The portfolio is long only and fully invested; the forecasts are those of one model (`xgb_long_drv_har`), the correlation is the EWMA's for every model.